# RAG Agent Pipeline -- Classroom Lab
## LlamaIndex + ChromaDB + OpenRouter + Ollama + ReAct Agent

---

| Step | Topic | Key Tool |
|------|-------|----------|
| 0 | Install dependencies | pip |
| 1 | Config: pick your backend | OpenRouter OR Ollama |
| 2 | Create sample PDF documents | fpdf2 |
| 3 | Load & inspect PDFs | LlamaIndex SimpleDirectoryReader |
| 4 | Node parsing (chunking) | SentenceSplitter |
| 5 | Embeddings -- two examples | HuggingFace (A) / Ollama (B) |
| 6 | Vector index (ChromaDB) | ChromaVectorStore |
| 7 | Query engine (basic RAG) | RetrieverQueryEngine |
| 8 | RAG Agent with tools | ReActAgent |
| 9 | Full demo | all components |
| 10 | Inspect internals | nodes, scores, similarity |
| 11 | Exercises | student tasks |

> **Model backends**
> - **Example A** -- OpenRouter (cloud, free tier)
> - **Example B** -- Ollama (100% local, offline)

---


## Step 0 -- Install All Dependencies

Run **once**, restart kernel when done.


In [ ]:
import subprocess, sys

packages = [
    "llama-index",
    "llama-index-core",
    "llama-index-llms-openai",
    "llama-index-embeddings-huggingface",
    "llama-index-vector-stores-chroma",
    "llama-index-readers-file",
    "chromadb",
    "sentence-transformers",
    "fpdf2",
    "pypdf",
    "rich",
    "httpx",
    "nest-asyncio",
]

for pkg in packages:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    icon = "OK" if r.returncode == 0 else "FAIL"
    print(f"{icon:4s}  {pkg}")

print("\nDone! Restart the kernel if this is your first run.")


---
## Step 1 -- Configuration

Pick ONE backend by setting `BACKEND`:

| Value | Runs where | Requires |
|-------|-----------|----------|
| `"openrouter"` | Cloud | Free key: https://openrouter.ai/keys |
| `"ollama"` | Your machine | Install ollama.ai, run `ollama pull llama3.2` |

> Embeddings **always run locally** (no API key needed for embeddings).


In [ ]:
import nest_asyncio
nest_asyncio.apply()

# === EDIT THIS BLOCK ===================================================

BACKEND = "openrouter"    # "openrouter"  or  "ollama"

# --- Example A: OpenRouter (cloud) -------------------------------------
OPENROUTER_API_KEY = "sk-or-v1-PASTE_YOUR_KEY_HERE"
OPENROUTER_MODEL   = "mistralai/mistral-7b-instruct"  # free model
# Other free models to try:
#   "meta-llama/llama-3.1-8b-instruct:free"
#   "google/gemma-3-27b-it:free"
#   "qwen/qwen3-8b:free"

# --- Example B: Ollama (local) -----------------------------------------
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL    = "llama3.2"   # ollama pull llama3.2

# --- Embedding model (local, free, ~90MB) ------------------------------
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"   # 384-dim
# Alternatives:
#   "all-MiniLM-L6-v2"         -- 384-dim, lighter
#   "BAAI/bge-base-en-v1.5"    -- 768-dim, more accurate

# =======================================================================
print(f"Backend    : {BACKEND}")
print(f"Embeddings : {EMBEDDING_MODEL}")


In [ ]:
from llama_index.llms.openai import OpenAI as LlamaOpenAI
from llama_index.core import Settings

if BACKEND == "openrouter":
    llm = LlamaOpenAI(
        model=OPENROUTER_MODEL,
        api_key=OPENROUTER_API_KEY,
        api_base="https://openrouter.ai/api/v1",
        temperature=0.1,
        max_tokens=1024,
        additional_kwargs={
            "headers": {"HTTP-Referer": "http://localhost", "X-Title": "RAG Lab"}
        }
    )
    print(f"OpenRouter LLM ready -- {OPENROUTER_MODEL}")

elif BACKEND == "ollama":
    llm = LlamaOpenAI(
        model=OLLAMA_MODEL,
        api_key="ollama",
        api_base=f"{OLLAMA_BASE_URL}/v1",
        temperature=0.1,
        max_tokens=1024,
    )
    print(f"Ollama LLM ready -- {OLLAMA_MODEL}")

else:
    raise ValueError(f"Unknown backend: {BACKEND}")

# Smoke test
resp = llm.complete("Reply with exactly: OK")
print(f"LLM test response: {resp.text.strip()}")


In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print(f"Loading embedding model: {EMBEDDING_MODEL} ...")
embed_model = HuggingFaceEmbedding(
    model_name=EMBEDDING_MODEL,
    trust_remote_code=True,
)

Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 256
Settings.chunk_overlap = 32

test_vec = embed_model.get_text_embedding("Hello")
print(f"Embedding model ready -- vector dim: {len(test_vec)}")


---
## Step 2 -- Create Sample PDF Documents

We generate **3 realistic PDFs** on AI-related topics.
In a real project you can drop any PDFs into `./docs/` and skip this step.


In [ ]:
from fpdf import FPDF
import os, textwrap

os.makedirs("docs", exist_ok=True)

DOCS_DATA = [
    (
        "ai_fundamentals.pdf",
        "AI Fundamentals -- A Primer",
        [
            (
                "What is Artificial Intelligence?",
                "Artificial Intelligence (AI) is the simulation of human-like intelligence in machines "
                "that can perform tasks such as visual perception, speech recognition, and decision-making. "
                "AI is divided into Narrow AI (task-specific) and General AI (human-level cognition). "
                "Machine Learning is a subset of AI where systems learn from data without explicit programming. "
                "Deep Learning uses multi-layer neural networks to learn hierarchical representations from data."
            ),
            (
                "Large Language Models",
                "LLMs are Transformer-based models trained on trillions of tokens of text. "
                "They demonstrate emergent abilities such as few-shot learning, chain-of-thought reasoning, "
                "and instruction following. The Transformer architecture introduced self-attention mechanisms "
                "in the 2017 paper Attention Is All You Need by Vaswani et al. "
                "Notable LLMs include GPT-4 (OpenAI), Claude (Anthropic), LLaMA 3 (Meta), and Mistral. "
                "LLMs can hallucinate -- generating plausible but incorrect content -- which is why RAG was created."
            ),
            (
                "Neural Network Architectures",
                "Convolutional Neural Networks (CNNs) excel at image classification and object detection. "
                "Recurrent Neural Networks (RNNs) and LSTMs handle sequential data like time series. "
                "Transformer networks use self-attention to model relationships in sequences in parallel. "
                "Encoder models like BERT are best for classification and understanding tasks. "
                "Decoder models like GPT are best for text generation tasks. "
                "Encoder-decoder models like T5 excel at translation and summarization."
            ),
        ]
    ),
    (
        "rag_deep_dive.pdf",
        "RAG -- Retrieval-Augmented Generation Deep Dive",
        [
            (
                "What is RAG?",
                "Retrieval-Augmented Generation (RAG) enhances LLM responses by grounding them in "
                "retrieved factual content from a knowledge base. Instead of relying solely on parametric "
                "memory (weights), RAG fetches relevant documents at inference time and includes them in "
                "the prompt context. This reduces hallucinations and enables up-to-date knowledge "
                "without retraining the model."
            ),
            (
                "RAG Pipeline -- Indexing Phase (Offline)",
                "Step 1: Load documents from PDFs, HTML pages, databases, or APIs. "
                "Step 2: Split into chunks using a text splitter, e.g. 256 tokens with 32 overlap. "
                "Step 3: Generate embeddings for each chunk using an embedding model. "
                "Step 4: Store embeddings and text in a vector database such as ChromaDB or Pinecone. "
                "This phase runs once offline and is updated when documents change."
            ),
            (
                "RAG Pipeline -- Retrieval Phase (Online)",
                "Step 1: Embed the user query with the same embedding model used during indexing. "
                "Step 2: Compute cosine similarity between the query vector and all stored vectors. "
                "Step 3: Retrieve the top-k most similar chunks, typically k=3 to 5. "
                "Step 4: Build a prompt: system instructions + retrieved context + user question. "
                "Step 5: The LLM generates an answer grounded in the retrieved context."
            ),
            (
                "Advanced RAG Techniques",
                "HyDE (Hypothetical Document Embeddings): generate a hypothetical answer, embed it, "
                "then retrieve similar real documents. Re-ranking: use a cross-encoder to reorder results. "
                "Query expansion: rewrite the query multiple ways and merge results. "
                "Sentence-window retrieval: index sentences but return surrounding windows for context. "
                "Parent-child retrieval: index small chunks but return large parent chunks."
            ),
        ]
    ),
    (
        "llamaindex_guide.pdf",
        "LlamaIndex -- Framework Guide",
        [
            (
                "What is LlamaIndex?",
                "LlamaIndex (formerly GPT Index) is an open-source data framework for building "
                "LLM-powered applications over custom data. It provides abstractions for data ingestion "
                "via Readers, indexing via VectorStoreIndex and SummaryIndex, querying via QueryEngine, "
                "and agent orchestration via ReActAgent. LlamaIndex integrates with 100+ data sources "
                "and 50+ LLM providers including OpenAI, Anthropic, Ollama, and OpenRouter."
            ),
            (
                "Core Components",
                "Document: the raw loaded content from any source. "
                "Node: a chunk of a Document with metadata and relationships to neighboring nodes. "
                "Embedding: a vector representation of a Node or query string. "
                "VectorStoreIndex: stores Nodes as vectors and enables similarity search. "
                "QueryEngine: takes a natural language question and returns a synthesized answer. "
                "Retriever: finds the most relevant Nodes given a query using vector similarity. "
                "NodePostprocessor: re-ranks or filters retrieved nodes before synthesis."
            ),
            (
                "ReActAgent in LlamaIndex",
                "LlamaIndex ReActAgent implements the Reason+Act loop for tool-using AI agents. "
                "The agent receives a question, reasons about which tool to use, calls the tool, "
                "observes the result, then repeats until it can produce a final answer. "
                "Tools can wrap QueryEngines for document search, Python functions for computation, "
                "or other agents for multi-agent systems. "
                "The agent uses structured output to parse Thought, Action, and Observation steps."
            ),
            (
                "ChromaDB Integration",
                "ChromaDB is a lightweight open-source vector database that runs locally with no server. "
                "LlamaIndex integrates with ChromaDB via the llama-index-vector-stores-chroma package. "
                "Collections persist to disk using DuckDB and Parquet files. "
                "ChromaDB supports cosine, dot-product, and L2 distance metrics. "
                "For production use, ChromaDB can be swapped for Pinecone, Weaviate, or Qdrant "
                "with minimal code changes in the LlamaIndex StorageContext."
            ),
        ]
    ),
]

def make_pdf(filename, title, sections):
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 18)
    pdf.cell(0, 12, title, ln=True, align="C")
    pdf.ln(6)
    for heading, body in sections:
        pdf.set_font("Helvetica", "B", 13)
        pdf.cell(0, 9, heading, ln=True)
        pdf.set_font("Helvetica", "", 11)
        for line in textwrap.wrap(body, 95):
            pdf.cell(0, 6, line, ln=True)
        pdf.ln(4)
    pdf.output(f"docs/{filename}")
    sz = os.path.getsize(f"docs/{filename}") // 1024
    print(f"  Created docs/{filename}  ({sz} KB)")

print("Generating sample PDFs ...")
for fname, title, sections in DOCS_DATA:
    make_pdf(fname, title, sections)

print(f"\nPDFs ready in ./docs/: {os.listdir("docs")}")


---
## Step 3 -- Load & Inspect PDFs with LlamaIndex

`SimpleDirectoryReader` auto-detects file types and routes to the correct loader.


In [ ]:
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader(
    input_dir="docs",
    required_exts=[".pdf"],
    recursive=False,
)
documents = reader.load_data()

print(f"Loaded {len(documents)} documents:\n")
for i, doc in enumerate(documents):
    fname = doc.metadata.get("file_name", f"doc_{i}")
    print(f"  [{i+1}] {fname}")
    print(f"       Chars  : {len(doc.text)}")
    print(f"       Preview: {doc.text[:100].replace(chr(10),' ')}...")
    print()


In [ ]:
# Inspect full text of first document
print('='*65)
print('FULL TEXT:', documents[0].metadata.get('file_name','doc_0'))
print('='*65)
print(documents[0].text[:1600])
print('\n...(truncated)')


---
## Step 4 -- Node Parsing (Chunking)

LlamaIndex splits Documents into **Nodes** -- the atomic unit indexed and retrieved.

    Document  -->  [Node1][Node2][Node3]...[NodeN]
                     chunk_size   chunk_overlap

Each Node stores: `text`, `metadata` (source, page), `node_id`, and prev/next relationships.


In [ ]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(
    chunk_size=256,
    chunk_overlap=32,
    paragraph_separator="\n\n",
)

nodes = splitter.get_nodes_from_documents(documents, show_progress=False)

print(f"Total nodes : {len(nodes)}")
print(f"Avg chars   : {sum(len(n.text) for n in nodes) // len(nodes)}")

from collections import Counter
src_counts = Counter(n.metadata.get("file_name","?") for n in nodes)
print("\nNodes per document:")
for src, cnt in src_counts.items():
    print(f"  {src}: {cnt} nodes")


In [ ]:
# Inspect one node in detail
n = nodes[4]
print("="*65)
print("NODE EXAMPLE (index 4)")
print("="*65)
print(f"Node ID : {n.node_id}")
print(f"Source  : {n.metadata.get('file_name','?')}")
print(f"Chars   : {len(n.text)}")
rels = list(n.relationships.items())
print(f"Relationships: {len(rels)}")
print(f"\nText:\n{n.text}")


---
## Step 5 -- Embeddings: Two Examples

### Example A -- HuggingFace (works with both OpenRouter and Ollama backends)
OpenRouter does not provide embedding endpoints, so we use a **local HuggingFace model**.
This is standard practice even when using a cloud LLM.

### Example B -- Ollama embeddings (fully local)
Ollama can serve both LLM and embedding models on the same machine.


In [ ]:
import numpy as np

# Example A: Local HuggingFace embeddings (already loaded in Step 1)
texts = [
    "RAG combines retrieval with language generation",
    "Retrieval-Augmented Generation grounds LLM answers in documents",  # semantically similar
    "The Eiffel Tower is located in Paris, France",                      # unrelated
]

vecs = [embed_model.get_text_embedding(t) for t in texts]

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"Embedding model  : {EMBEDDING_MODEL}")
print(f"Vector dimension : {len(vecs[0])}")
print()
print("Cosine similarity:")
labels = ["S1 (RAG)", "S2 (similar)", "S3 (unrelated)"]
for i in range(3):
    for j in range(i, 3):
        sim = cosine(vecs[i], vecs[j])
        note = " <- should be HIGH" if i==0 and j==1 else (" <- should be LOW" if j==2 and i<2 else "")
        print(f"  {labels[i]:20s} vs {labels[j]:20s}  =  {sim:.4f}{note}")


In [ ]:
# Example B: Ollama embedding model code pattern
# Activate this block if BACKEND == 'ollama'

print("Example B -- Ollama Embedding Pattern")
print("-"*55)
ollama_code = '''
# Install:
# pip install llama-index-embeddings-ollama

from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core import Settings

ollama_embed = OllamaEmbedding(
    model_name="nomic-embed-text",   # ollama pull nomic-embed-text
    base_url="http://localhost:11434",
)

Settings.embed_model = ollama_embed

vec = ollama_embed.get_text_embedding("Hello from Ollama!")
print(f"Dimension: {len(vec)}")   # nomic-embed-text = 768
'''
print(ollama_code)

# Auto-activate if ollama backend is selected
if BACKEND == 'ollama':
    try:
        from llama_index.embeddings.ollama import OllamaEmbedding
        embed_model = OllamaEmbedding(
            model_name="nomic-embed-text",
            base_url=OLLAMA_BASE_URL,
        )
        Settings.embed_model = embed_model
        t = embed_model.get_text_embedding("test")
        print(f"Switched to Ollama embeddings -- dim: {len(t)}")
    except Exception as e:
        print(f"Could not load Ollama embeddings ({e}). Keeping HuggingFace.")
else:
    print("Using HuggingFace embeddings (standard for OpenRouter).")


---
## Step 6 -- Vector Index with ChromaDB

    Nodes --> embed_model --> vectors --> ChromaDB (on disk)
                                              ^
                                    VectorStoreIndex

ChromaDB runs fully locally -- no server, no cloud account, no cost.


In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext, VectorStoreIndex

CHROMA_PATH = "./chroma_rag_lab"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Remove old collection when re-running the notebook
try:
    chroma_client.delete_collection("rag_lab_docs")
    print("Cleared previous collection.")
except:
    pass

collection = chroma_client.get_or_create_collection(
    name="rag_lab_docs",
    metadata={"hnsw:space": "cosine"}
)
print(f"ChromaDB collection ready: {CHROMA_PATH}")

# Wrap with LlamaIndex
vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Build the index -- embeds every node and stores vectors
print("Embedding nodes and storing in ChromaDB ...")
index = VectorStoreIndex(
    nodes=nodes,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True,
)
print(f"\nIndex built -- {collection.count()} vectors in ChromaDB")


In [ ]:
# Inspect ChromaDB collection contents
results = collection.get(limit=5, include=["documents","metadatas"])
print("ChromaDB snapshot (first 5 entries):\n")
for i, (doc, meta) in enumerate(zip(results["documents"], results["metadatas"])):
    print(f"  [{i+1}] Source: {meta.get("file_name","?")}")
    print(f"       Text  : {doc[:100]}...")
    print()
print(f"Total vectors in collection: {collection.count()}")


---
## Step 7 -- Query Engine (Basic RAG)

The QueryEngine is the simplest end-to-end RAG pipeline:

    Question --> embed --> ChromaDB top-k --> Prompt --> LLM --> Answer


In [ ]:
from llama_index.core import PromptTemplate
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.response_synthesizers import get_response_synthesizer

# Custom RAG prompt
RAG_PROMPT_TMPL = (
    "You are an expert AI tutor. Answer the question using ONLY the context provided.\n"
    "If the context does not contain the answer, say: The documents do not cover this.\n"
    "Always cite which document your answer comes from.\n\n"
    "CONTEXT:\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n\n"
    "QUESTION: {query_str}\n\n"
    "ANSWER:"
)
rag_prompt = PromptTemplate(RAG_PROMPT_TMPL)

# Build retriever (top-3 chunks)
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=3,
)

# Build response synthesizer
response_synthesizer = get_response_synthesizer(
    llm=llm,
    text_qa_template=rag_prompt,
    response_mode="compact",
)

# Assemble query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)

print("QueryEngine ready")
print(f"  Retriever  : VectorIndexRetriever (top_k=3)")
print(f"  Synthesizer: compact")
print(f"  LLM        : {BACKEND}")


In [ ]:
# Helper: ask with source node display
def ask(question, engine=None):
    if engine is None:
        engine = query_engine
    print(f"\n{'='*65}")
    print(f"QUESTION: {question}")
    print('='*65)
    response = engine.query(question)
    print(f"\nANSWER:\n{response.response}")
    print(f"\nSOURCE NODES ({len(response.source_nodes)}):")
    for i, sn in enumerate(response.source_nodes, 1):
        src   = sn.node.metadata.get('file_name', '?')
        score = f"{sn.score:.4f}" if sn.score else "n/a"
        print(f"  [{i}] {src}  score={score}")
        print(f"       {sn.node.text[:120]}...")

ask("What is Retrieval-Augmented Generation and why does it reduce hallucinations?")


In [ ]:
ask("What are the two phases of a RAG pipeline? Describe each step.")


In [ ]:
ask("What is LlamaIndex and what are its core components?")


In [ ]:
# Test with a question NOT covered by documents
ask("What is the population of Tokyo?")


---
## Step 8 -- RAG Agent with ReAct + Multiple Tools

The **ReActAgent** is more powerful than a QueryEngine.
It can reason about which tool to call, call tools multiple times, and combine results.

    User Question
         |
    Thought: I need to search the documents first...
         |
    Action  --> document_search(query)
         |
    Observation: [retrieved chunks]
         |
    Thought: Now I need to calculate...
         |
    Action  --> calculator(expression)
         |
    Observation: 3,840,000
         |
    Final Answer


In [ ]:
from llama_index.core.tools import QueryEngineTool, FunctionTool, ToolMetadata
from llama_index.core.agent import ReActAgent
import math

# ---- Tool 1: Document Search --------------------------------------------
doc_search_tool = QueryEngineTool(
    query_engine=query_engine,
    metadata=ToolMetadata(
        name="document_search",
        description=(
            "Search the knowledge base for information about: "
            "AI fundamentals, machine learning, deep learning, Transformers, LLMs, "
            "RAG pipeline (indexing and retrieval phases), advanced RAG techniques, "
            "LlamaIndex framework and components, ChromaDB, and ReActAgent. "
            "Use this tool for any question about these topics. "
            "Input: a natural language question."
        ),
    ),
)

# ---- Tool 2: Calculator -------------------------------------------------
def calculator(expression: str) -> str:
    """Evaluate a Python math expression. Input: expression like '384 * 10000' or 'sqrt(2)'.     """
    try:
        safe = {k: getattr(math, k) for k in dir(math) if not k.startswith('_')}
        safe.update({'abs': abs, 'round': round, 'int': int, 'float': float})
        result = eval(expression, {"__builtins__": {}}, safe)
        return f"Result: {result}"
    except Exception as e:
        return f"Math error: {e}"

calc_tool = FunctionTool.from_defaults(
    fn=calculator,
    name="calculator",
    description="Evaluate mathematical expressions. Input: a Python expression like '384 * 10000'."
)

# ---- Tool 3: Glossary ---------------------------------------------------
GLOSSARY = {
    "rag": "Retrieval-Augmented Generation: grounds LLM outputs in retrieved documents to reduce hallucination.",
    "llm": "Large Language Model: a deep learning model trained on massive text corpora.",
    "embedding": "A dense numerical vector encoding the semantic meaning of text.",
    "vector database": "A database optimised for storing and searching high-dimensional vectors.",
    "cosine similarity": "Similarity metric based on the angle between two vectors. Range -1 to 1.",
    "chunking": "Splitting documents into smaller pieces (nodes/chunks) for indexing.",
    "top-k": "Retrieving the k items with highest similarity score from vector search.",
    "hallucination": "LLM generating plausible but factually incorrect content.",
    "react": "Reason+Act: an agent pattern alternating between reasoning and tool calls.",
    "llamaindex": "Open-source Python framework for building LLM applications over custom data.",
    "chromadb": "Open-source local vector database using DuckDB and Parquet files.",
    "transformer": "Neural architecture using self-attention, the foundation of all modern LLMs.",
    "node": "In LlamaIndex: a document chunk with metadata and inter-node relationships.",
    "hyde": "Hypothetical Document Embeddings: embed a hypothetical answer to improve retrieval.",
    "sentence-transformers": "HuggingFace library providing pre-trained embedding models.",
}

def glossary_lookup(term: str) -> str:
    """Look up an AI/ML technical term definition. Input: the term (case-insensitive)."""
    t = term.lower().strip()
    if t in GLOSSARY:
        return f"{term}: {GLOSSARY[t]}"
    matches = {k: v for k, v in GLOSSARY.items() if t in k or k in t}
    if matches:
        return "\n".join(f"{k}: {v}" for k, v in matches.items())
    return f"'{term}' not found. Available: {', '.join(GLOSSARY.keys())}"

glossary_tool = FunctionTool.from_defaults(
    fn=glossary_lookup,
    name="glossary",
    description="Look up definitions of AI/ML technical terms like RAG, embedding, cosine similarity, etc."
)

# ---- Tool 4: Pipeline Stats ---------------------------------------------
def pipeline_stats(query: str = '') -> str:
    """Return statistics about the loaded documents and vector index."""
    dim = len(embed_model.get_text_embedding('x'))
    lines = [
        f"Documents loaded  : {len(documents)}",
        f"Total nodes       : {len(nodes)}",
        f"Embedding model   : {EMBEDDING_MODEL}",
        f"Vector dimension  : {dim}",
        f"ChromaDB vectors  : {collection.count()}",
        f"LLM backend       : {BACKEND}",
    ]
    for doc in documents:
        lines.append(f"  File: {doc.metadata.get('file_name','?')}  ({len(doc.text)} chars)")
    return "\n".join(lines)

stats_tool = FunctionTool.from_defaults(
    fn=pipeline_stats,
    name="pipeline_stats",
    description="Return stats about the documents, node count, embedding dimensions, and vector index."
)

TOOLS = [doc_search_tool, calc_tool, glossary_tool, stats_tool]
print("Tools registered:")
for t in TOOLS:
    print(f"  {t.metadata.name:20s}  {t.metadata.description[:60]}...")


In [ ]:
# Build the ReActAgent
agent = ReActAgent.from_tools(
    tools=TOOLS,
    llm=llm,
    verbose=True,   # prints Thought / Action / Observation
    max_iterations=8,
)

print("ReActAgent ready!")
print(f"  Tools     : {[t.metadata.name for t in TOOLS]}")
print(f"  Max iters : 8")
print(f"  LLM       : {BACKEND}")


---
## Step 9 -- Full Pipeline Demo

Watch the **Thought -> Action -> Observation** loop.


In [ ]:
def run_agent(question):
    print('\n' + '='*65)
    print(f'QUESTION: {question}')
    print('='*65 + '\n')
    try:
        result = agent.chat(question)
        print('\n' + '-'*65)
        print(f'FINAL ANSWER:\n{result.response}')
        print('-'*65)
    except Exception as e:
        print(f'Agent error: {e}')
        raise


In [ ]:
# Demo 1: Pure document lookup
run_agent("What are the two phases of a RAG pipeline? Give details on each.")


In [ ]:
# Demo 2: Glossary + document search
run_agent("Define cosine similarity, then explain exactly how it is used in the RAG retrieval phase.")


In [ ]:
# Demo 3: Stats + calculator
run_agent(
    "How many vectors are in the ChromaDB index? "
    "If each vector has 384 dimensions stored as 32-bit floats (4 bytes each), "
    "how many bytes total are stored? Convert to megabytes."
)


In [ ]:
# Demo 4: Multi-document reasoning
run_agent(
    "Compare how the LlamaIndex guide describes the ReActAgent "
    "with the general concept of AI agents. What is the connection?"
)


In [ ]:
# Demo 5: Out-of-scope question
run_agent("Who won the FIFA World Cup in 2022?")


---
## Step 10 -- Inspect Internals

Understand what is happening under the hood.


In [ ]:
# 10A: Raw retrieval with similarity scores
from llama_index.core.retrievers import VectorIndexRetriever

query_text = "What is the indexing phase of RAG?"
ret_inspect = VectorIndexRetriever(index=index, similarity_top_k=5)
raw_nodes = ret_inspect.retrieve(query_text)

print(f"Raw retrieval for: '{query_text}'")
print("="*65)
for i, rn in enumerate(raw_nodes, 1):
    src = rn.node.metadata.get('file_name', '?')
    score = rn.score if rn.score else 0
    print(f"\n[{i}] Score: {score:.4f}  Source: {src}")
    print(f"    {rn.node.text[:200]}...")


In [ ]:
# 10B: Direct ChromaDB query (bypass LlamaIndex)
query_emb = embed_model.get_text_embedding("vector similarity search")
chroma_res = collection.query(
    query_embeddings=[query_emb],
    n_results=3,
    include=["documents","metadatas","distances"]
)

print("Direct ChromaDB query (distance = 1 - cosine_similarity):\n")
for i, (doc, meta, dist) in enumerate(
    zip(chroma_res["documents"][0], chroma_res["metadatas"][0], chroma_res["distances"][0]), 1
):
    sim = 1 - dist
    print(f"[{i}] Cosine sim: {sim:.4f}  Source: {meta.get('file_name','?')}")
    print(f"    {doc[:200]}...\n")


In [ ]:
# 10C: Embedding similarity heatmap
import numpy as np

phrases = [
    "RAG retrieval phase",
    "LlamaIndex query engine",
    "cosine similarity vectors",
    "neural network transformer",
    "ChromaDB vector store",
    "apple pie baking recipe",
]

vecs = np.array([embed_model.get_text_embedding(p) for p in phrases])
norms = np.linalg.norm(vecs, axis=1, keepdims=True)
sim_mat = vecs @ vecs.T / (norms @ norms.T)

print("Embedding Similarity Matrix (cosine, 1.0 = identical):\n")
header = "".join(f"{i+1:>7}" for i in range(len(phrases)))
print(f"{"":>30}{header}")
for i, phrase in enumerate(phrases):
    row = "".join(f"{sim_mat[i,j]:>7.3f}" for j in range(len(phrases)))
    print(f"{phrase[:28]:30s}{row}")

print("\nKey:")
for i, p in enumerate(phrases):
    print(f"  {i+1}: {p}")


In [ ]:
# 10D: Node relationship chain
from collections import defaultdict

print("Node chains (first 3 nodes per document):\n")
doc_nodes = defaultdict(list)
for n in nodes:
    doc_nodes[n.metadata.get('file_name','?')].append(n)

for src, dnodes in doc_nodes.items():
    print(f"Document: {src}")
    for n in dnodes[:3]:
        nid = n.node_id[:10]
        print(f"   [{nid}]  chars={len(n.text)}")
        print(f"   Text: {n.text[:80]}...")
    print()


---
## Step 11 -- Exercises

Try these before looking at any solutions!

---
### Exercise 1 -- Add a new topic PDF
Add a document on any topic, re-index it, and query the agent.


In [ ]:
# TODO: Write content on a topic of your choice.
# Then generate the PDF, reload, and add to the index.

from fpdf import FPDF
import textwrap, os

MY_TOPIC_TITLE   = "Quantum Computing Basics"
MY_TOPIC_FILE    = "docs/quantum_computing.pdf"
MY_TOPIC_CONTENT = """
Quantum computing uses quantum mechanical phenomena -- superposition and entanglement -- to
process information. A qubit can be simultaneously 0 and 1 until measured (superposition),
enabling massive parallelism. Entanglement links two qubits so the state of one instantly
affects the other. Quantum gates manipulate qubits analogously to classical logic gates.
Shor algorithm uses quantum computing to factor large integers exponentially faster than
classical computers, threatening RSA cryptography. Grover algorithm speeds up unstructured
search quadratically. Current quantum computers from IBM and Google are NISQ devices
(Noisy Intermediate-Scale Quantum) with 100-1000 qubits but significant error rates.
Quantum error correction and fault tolerance are active research areas.
"""

pdf_new = FPDF()
pdf_new.add_page()
pdf_new.set_font("Helvetica", "B", 16)
pdf_new.cell(0, 10, MY_TOPIC_TITLE, ln=True)
pdf_new.set_font("Helvetica", "", 11)
for line in textwrap.wrap(MY_TOPIC_CONTENT.strip(), 95):
    pdf_new.cell(0, 6, line, ln=True)
pdf_new.output(MY_TOPIC_FILE)
print(f"Created {MY_TOPIC_FILE}")

# Load and add to index
from llama_index.core import SimpleDirectoryReader
new_docs = SimpleDirectoryReader(input_files=[MY_TOPIC_FILE]).load_data()
new_nodes = splitter.get_nodes_from_documents(new_docs)
index.insert_nodes(new_nodes)
print(f"Added {len(new_nodes)} new nodes. Total in ChromaDB: {collection.count()}")

# Now query!
ask("What is Shor algorithm and why is it important for cryptography?")


---
### Exercise 2 -- Compare chunk sizes
Rebuild the index with chunk_size=128 vs 512. Which gives better retrieval scores?


In [ ]:
def build_experiment_engine(chunk_sz, overlap):
    from llama_index.core.node_parser import SentenceSplitter
    from llama_index.core import StorageContext, VectorStoreIndex
    from llama_index.vector_stores.chroma import ChromaVectorStore
    from llama_index.core.retrievers import VectorIndexRetriever
    from llama_index.core.query_engine import RetrieverQueryEngine
    from llama_index.core.response_synthesizers import get_response_synthesizer

    sp = SentenceSplitter(chunk_size=chunk_sz, chunk_overlap=overlap)
    nds = sp.get_nodes_from_documents(documents)

    cname = f"exp_{chunk_sz}"
    try: chroma_client.delete_collection(cname)
    except: pass
    coll = chroma_client.get_or_create_collection(cname, metadata={"hnsw:space": "cosine"})
    vs = ChromaVectorStore(chroma_collection=coll)
    sc = StorageContext.from_defaults(vector_store=vs)
    idx = VectorStoreIndex(nodes=nds, storage_context=sc, embed_model=embed_model)
    qe = RetrieverQueryEngine(
        retriever=VectorIndexRetriever(index=idx, similarity_top_k=3),
        response_synthesizer=get_response_synthesizer(llm=llm, text_qa_template=rag_prompt)
    )
    return qe, len(nds)

test_question = "What are advanced RAG techniques?"
print(f"Test question: {test_question}\n")
print(f"{"chunk_size":>12}  {"overlap":>8}  {"nodes":>6}  {"top_score":>10}  answer_preview")
print("-"*80)

for cs, ov in [(128, 16), (256, 32), (512, 64), (1024, 128)]:
    qe_exp, n = build_experiment_engine(cs, ov)
    resp = qe_exp.query(test_question)
    top_score = resp.source_nodes[0].score if resp.source_nodes else 0
    preview = resp.response[:80].replace('\n', ' ')
    print(f"{cs:>12}  {ov:>8}  {n:>6}  {top_score:>10.4f}  {preview}...")


---
### Exercise 3 -- Add a custom tool
Implement your own tool and add it to the agent.


In [ ]:
from llama_index.core.tools import FunctionTool

# TODO: Replace this template with your own tool logic.
# Ideas: word counter, unit converter, date calculator, sentiment analyzer

def text_stats(text: str) -> str:
    """Compute text statistics: word count, character count, sentence count."""
    words = len(text.split())
    chars = len(text)
    sentences = text.count('.') + text.count('!') + text.count('?')
    unique_words = len(set(w.lower().strip('.,!?') for w in text.split()))
    return (
        f"Text statistics for input ({chars} chars):\n"
        f"  Words        : {words}\n"
        f"  Unique words : {unique_words}\n"
        f"  Sentences    : {sentences}\n"
        f"  Avg word len : {chars / max(words,1):.1f} chars"
    )

my_tool = FunctionTool.from_defaults(
    fn=text_stats,
    name="text_stats",
    description="Count words, characters, sentences in a text. Input: any string."
)

extended_tools = TOOLS + [my_tool]
agent_v2 = ReActAgent.from_tools(
    tools=extended_tools, llm=llm, verbose=True, max_iterations=8
)

print(f"Extended agent tools: {[t.metadata.name for t in extended_tools]}")
result = agent_v2.chat(
    "Count the words and sentences in this text: "
    "The quick brown fox jumps over the lazy dog. "
    "Pack my box with five dozen liquor jugs."
)
print(f"\nAnswer: {result.response}")


---
## Summary & Architecture Map

### Full pipeline built in this lab:

    PDF Files
    (ai_fundamentals.pdf, rag_deep_dive.pdf, llamaindex_guide.pdf)
           |
    SimpleDirectoryReader
           |
    SentenceSplitter (chunk_size=256, overlap=32)
           |
          Nodes
           |
    HuggingFaceEmbedding  [OR Ollama: nomic-embed-text]
    BAAI/bge-small-en-v1.5 (384-dim)
           |
        ChromaDB
    (persisted to ./chroma_rag_lab/)
           |
    VectorStoreIndex
           |
    --------+----------
    |                 |
    QueryEngine       ReActAgent
    (basic RAG)       (Tools: doc_search, calc, glossary, stats)
    |                 |
    --------+----------
           |
     LLM (OpenRouter or Ollama)
           |
      Final Answer

### Technology Stack

| Layer | Tool | Notes |
|-------|------|-------|
| Framework | **LlamaIndex** | Orchestration, nodes, agents |
| Vector DB | **ChromaDB** | Local, free, persistent |
| Embeddings A | **BAAI/bge-small-en-v1.5** | HuggingFace, no API |
| Embeddings B | **nomic-embed-text** | Ollama local |
| LLM (cloud) | **OpenRouter** | Free tier, 200+ models |
| LLM (local) | **Ollama** | 100% offline |
| PDF creation | **fpdf2** | Generate sample docs |
| PDF reading | **LlamaIndex / pypdf** | Auto-detected |
| Agent pattern | **ReAct** | Reason + Act loop |

### Next Steps
- Add web search tool (Tavily / DuckDuckGo)
- Add conversation memory (ChatMemoryBuffer)
- Evaluate with RAGAS (faithfulness, answer relevancy, context recall)
- Deploy as Streamlit or FastAPI app
- Try HyDE retrieval for better accuracy
- Explore multi-agent patterns (sub-agents)


In [ ]:
# Optional: clean up generated files
# import shutil
# shutil.rmtree('./chroma_rag_lab', ignore_errors=True)
# shutil.rmtree('./docs', ignore_errors=True)
# print('Cleaned up')

print("Congratulations! You completed the RAG Agent Classroom Lab.")
print(f"  Documents loaded  : {len(documents)}")
print(f"  Nodes indexed     : {len(nodes)}")
print(f"  ChromaDB vectors  : {collection.count()}")
print(f"  Agent tools       : {len(TOOLS)}")
